In [23]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats

In [24]:
X = pd.read_pickle('../data/fuzzy_x_20240106_values.pkl')
shap_values = np.load('../data/fuzzy_cat_20240106_shap_values.npy')
shap_interaction_values = np.load('../data/fuzzy_cat_20240106_shap_interaction_values.npy')

In [25]:
X = X.round(4)
shap_values = np.round(shap_values, 4)
shap_interaction_values = np.round(shap_interaction_values, 4)

In [26]:
variable_A_name = "I_12(mu)"
variable_A_index = X.columns.get_loc(variable_A_name)
variable_B_name = "Tipo P"
variable_B_index = X.columns.get_loc(variable_B_name)

In [27]:
from cgt_perezsechi.exploration.sampling import sample_size_kruskal_wallis

sample_size = sample_size_kruskal_wallis(X, shap_values, variable_A_name, acceleration=1)
if sample_size < len(X) / 10:
    sample_size = len(X) // 10

In [28]:
from cgt_perezsechi.manipulation.coding import code_interval

X_coded = code_interval(X, variable_A_name, sample_size)
df_A = pd.DataFrame({
    variable_A_name: X_coded[f'{variable_A_name}_interval'],
    'shap': shap_values[:, X.columns.get_loc(variable_A_name)]
})

In [29]:
df_B = pd.DataFrame({
    variable_B_name: X[variable_B_name],
    'shap': shap_values[:, X.columns.get_loc(variable_B_name)]
})

## Construcción de matrices de adyacencia

In [30]:
shap_interaction_values_A_B = shap_interaction_values[:, variable_A_index, variable_B_index]
df_interaction = pd.DataFrame({
    variable_A_name: X_coded[f'{variable_A_name}_interval'],
    variable_B_name: X_coded[variable_B_name],
    'shap_interaction': shap_interaction_values_A_B
})
df_interaction_grouped = df_interaction.groupby([variable_A_name, variable_B_name]).agg(["mean"]).reset_index()
df_interaction_grouped.columns = [variable_A_name, variable_B_name, 'shap_interaction']
df_interaction_grouped

,I_12(mu),Tipo P,shap_interaction
0,"[-0.0,0.0539]",0,4967.001252
1,"[-0.0,0.0539]",1,-8092.862699
2,"[0.0546,0.1375]",0,4387.868147
3,"[0.0546,0.1375]",1,-6789.789472
4,"[0.1384,0.225]",0,1511.922570
5,"[0.1384,0.225]",1,-2277.878290
6,"[0.2251,0.314]",0,-750.171830
7,"[0.2251,0.314]",1,1180.412153
8,"[0.3146,0.5707]",0,-3793.762479
9,"[0.3146,0.5707]",1,6106.178311


In [31]:
unique_A_values = df_interaction_grouped[variable_A_name].unique()
unique_B_values = df_interaction_grouped[variable_B_name].unique()
variable_A_column_names = [f"{variable_A_name}_{value}" for value in unique_A_values]
variable_B_column_names = [f"{variable_B_name}_{value}" for value in unique_B_values]
column_names = variable_A_column_names + variable_B_column_names
r = pd.DataFrame(
    columns=column_names,
)

for A_value in unique_A_values:
    for B_value in unique_B_values:
        index_A = f"{variable_A_name}_{A_value}" 
        index_B = f"{variable_B_name}_{B_value}" 
        shap_interaction = df_interaction_grouped[
            (df_interaction_grouped[variable_A_name] == A_value) &
            (df_interaction_grouped[variable_B_name] == B_value)
        ]['shap_interaction'].values[0]
        r.loc[index_A, index_B] = shap_interaction
        r.loc[index_B, index_A] = shap_interaction

r.replace(np.nan, 0, inplace=True)
r_1 = r

C:\Users\secci\AppData\Local\Temp\ipykernel_17844\2317974099.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  r.replace(np.nan, 0, inplace=True)


In [32]:
r_1

,"I_12(mu)_[-0.0,0.0539]","I_12(mu)_[0.0546,0.1375]","I_12(mu)_[0.1384,0.225]","I_12(mu)_[0.2251,0.314]","I_12(mu)_[0.3146,0.5707]","I_12(mu)_[0.5725,0.934]",Tipo P_0,Tipo P_1
"I_12(mu)_[-0.0,0.0539]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,4967.001252,-8092.862699
Tipo P_0,4967.001252,4387.868147,1511.92257,-750.171830,-3793.762479,-21638.571029,0.000000,0.000000
Tipo P_1,-8092.862699,-6789.789472,-2277.87829,1180.412153,6106.178311,61193.159109,0.000000,0.000000
"I_12(mu)_[0.0546,0.1375]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,4387.868147,-6789.789472
"I_12(mu)_[0.1384,0.225]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1511.922570,-2277.878290
"I_12(mu)_[0.2251,0.314]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-750.171830,1180.412153
"I_12(mu)_[0.3146,0.5707]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-3793.762479,6106.178311
"I_12(mu)_[0.5725,0.934]",0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-21638.571029,61193.159109


## Representación en grafo de la interacción entre valores de las variables "age" y "sex_isFemale"

In [33]:
df_A_stats = df_A.groupby(variable_A_name).agg(['mean']).reset_index()
df_A_stats.columns = [variable_A_name, 'shap']
df_B_stats = df_B.groupby(variable_B_name).agg(['mean']).reset_index()
df_B_stats.columns = [variable_B_name, 'shap']

In [34]:
df_A_stats

,I_12(mu),shap
0,"[-0.0,0.0539]",-12867.156421
1,"[0.0546,0.1375]",-10596.261485
2,"[0.1384,0.225]",-2372.440734
3,"[0.2251,0.314]",161.958353
4,"[0.3146,0.5707]",8396.726744
5,"[0.5725,0.934]",77220.878737


In [35]:
psi_1 = pd.DataFrame(
    columns=['value'],
)

for index, row in df_A_stats.iterrows():
    psi_1.loc[f'{variable_A_name}_{row[variable_A_name]}', 'value'] = row['shap']
for index, row in df_B_stats.iterrows():
    psi_1.loc[f'{variable_B_name}_{row[variable_B_name]:.0f}', 'value'] = row['shap']

psi_1.index.name = None
psi_1

,value
"I_12(mu)_[-0.0,0.0539]",-12867.156421
"I_12(mu)_[0.0546,0.1375]",-10596.261485
"I_12(mu)_[0.1384,0.225]",-2372.440734
"I_12(mu)_[0.2251,0.314]",161.958353
"I_12(mu)_[0.3146,0.5707]",8396.726744
"I_12(mu)_[0.5725,0.934]",77220.878737
Tipo P_0,-12265.838634
Tipo P_1,20139.475231


In [36]:
from cgt_perezsechi.manipulation.norm import normalize_psi, normalize_r

psi_2 = normalize_psi(psi_1)
r_2 = normalize_r(r_1)

c:\Workspace\fuzzieee-2025\fuzzieee-2025-python\.venv\Lib\site-packages\cgt_perezsechi\manipulation\norm.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  max_edge_width = r.applymap(lambda x: abs(x)).max().max()
c:\Workspace\fuzzieee-2025\fuzzieee-2025-python\.venv\Lib\site-packages\cgt_perezsechi\manipulation\norm.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  r = r.copy().applymap(lambda x: x / max_edge_width)


In [37]:
import regex as re
# Create new column names with TeX expressions
r_2_cols = r_2.columns.str.replace('I_12(mu)_', r'I_{12}(\mu) \in ').str.replace(r'^|$', '$', regex=True)
r_2_cols = r_2_cols.str.replace('Tipo P_1', 'Type = Innovative') 
r_2_cols = r_2_cols.str.replace('Tipo P_0', 'Type = Non-Innovative') 
r_2.columns = r_2_cols

# Update index as well
r_2.index = r_2.index.str.replace('I_12(mu)_', r'I_{12}(\mu) \in ').str.replace(r'^|$', '$', regex=True)
r_2.index = r_2.index.str.replace('Tipo P_1', 'Type = Innovative')
r_2.index = r_2.index.str.replace('Tipo P_0', 'Type = Non-Innovative')

# Update psi_2 index
psi_2.index = psi_2.index.str.replace('I_12(mu)_', r'I_{12}(\mu) \in ').str.replace(r'^|$', '$', regex=True)
psi_2.index = psi_2.index.str.replace('Tipo P_1', 'Type = Innovative')
psi_2.index = psi_2.index.str.replace('Tipo P_0', 'Type = Non-Innovative')

In [38]:
r_2

,"$I_{12}(\mu) \in [-0.0,0.0539]$","$I_{12}(\mu) \in [0.0546,0.1375]$","$I_{12}(\mu) \in [0.1384,0.225]$","$I_{12}(\mu) \in [0.2251,0.314]$","$I_{12}(\mu) \in [0.3146,0.5707]$","$I_{12}(\mu) \in [0.5725,0.934]$",$Type = Non-Innovative$,$Type = Innovative$
"$I_{12}(\mu) \in [-0.0,0.0539]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.081169,-0.132251
$Type = Non-Innovative$,0.081169,0.071705,0.024707,-0.012259,-0.061997,-0.353611,0.000000,0.000000
$Type = Innovative$,-0.132251,-0.110957,-0.037224,0.019290,0.099785,1.000000,0.000000,0.000000
"$I_{12}(\mu) \in [0.0546,0.1375]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.071705,-0.110957
"$I_{12}(\mu) \in [0.1384,0.225]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.024707,-0.037224
"$I_{12}(\mu) \in [0.2251,0.314]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.012259,0.019290
"$I_{12}(\mu) \in [0.3146,0.5707]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.061997,0.099785
"$I_{12}(\mu) \in [0.5725,0.934]$",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.353611,1.000000


In [39]:
psi_2

,value
"$I_{12}(\mu) \in [-0.0,0.0539]$",-0.166628
"$I_{12}(\mu) \in [0.0546,0.1375]$",-0.137220
"$I_{12}(\mu) \in [0.1384,0.225]$",-0.030723
"$I_{12}(\mu) \in [0.2251,0.314]$",0.002097
"$I_{12}(\mu) \in [0.3146,0.5707]$",0.108736
"$I_{12}(\mu) \in [0.5725,0.934]$",1.000000
$Type = Non-Innovative$,-0.158841
$Type = Innovative$,0.260803


In [41]:
from cgt_perezsechi.visualization.graph import draw


positive_alpha = 0
negative_alpha = 0
positive_beta = 0
negative_beta = 0
draw(
    psi=psi_2,
    r=r_2,
    positive_alpha=positive_alpha,
    negative_alpha=negative_alpha,
    positive_beta=positive_beta,
    negative_beta=negative_beta,
    positive_color="#2084d6",
    negative_color="#df205d",
    layout="shell",
    node_label_size_limit=5000,
    node_pos={
        r'$Type = Non-Innovative$': (350, 500),
        r'$Type = Innovative$': (350, 200),
        r'$I_{12}(\mu) \in [-0.0,0.0539]$': (90, 600),
        r'$I_{12}(\mu) \in [0.0546,0.1375]$': (90, 500),
        r'$I_{12}(\mu) \in [0.1384,0.225]$': (90, 400),
        r'$I_{12}(\mu) \in [0.2251,0.314]$': (90, 300),
        r'$I_{12}(\mu) \in [0.3146,0.5707]$': (90, 200),
        r'$I_{12}(\mu) \in [0.5725,0.934]$': (90, 100),
    },
    label_pos={
        r'$Type = Non-Innovative$': (420, 500),
        r'$Type = Innovative$': (410, 200),
        r'$I_{12}(\mu) \in [-0.0,0.0539]$': (0, 600),
        r'$I_{12}(\mu) \in [0.0546,0.1375]$': (0, 500),
        r'$I_{12}(\mu) \in [0.1384,0.225]$': (0, 400),
        r'$I_{12}(\mu) \in [0.2251,0.314]$': (0, 300),
        r'$I_{12}(\mu) \in [0.3146,0.5707]$': (0, 200),
        r'$I_{12}(\mu) \in [0.5725,0.934]$': (0, 100),
    },
    plot_margin=0.1,
    label_color="#000000",
    output_path="../output/kw_I_{12}_fuzzy_cat_20240106.jpg",
)

c:\Workspace\fuzzieee-2025\fuzzieee-2025-python\.venv\Lib\site-packages\cgt_perezsechi\visualization\graph.py:54: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  r = r.copy().applymap(filter_edge)
c:\Workspace\fuzzieee-2025\fuzzieee-2025-python\.venv\Lib\site-packages\cgt_perezsechi\visualization\graph.py:55: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  adjacency = r.copy().applymap(lambda x: 1 if x != 0 else 0)
c:\Workspace\fuzzieee-2025\fuzzieee-2025-python\.venv\Lib\site-packages\cgt_perezsechi\visualization\graph.py:240: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saving the graph to ../output/kw_I_{12}_fuzzy_cat_20240106.jpg
